# iMessages API Test Notebook

Test the `/v1/imessages` endpoint that returns iMessages grouped by thread.

The endpoint accepts a `from_date` parameter (ISO 8601 datetime) and returns:
- Messages grouped by thread
- Thread participants (resolved names from contacts)
- Threads ordered by earliest message (ascending)
- Messages within each thread ordered by date (ascending)
- Sender names resolved from contacts (fallback to identifier)


In [1]:
import os
from datetime import datetime, timedelta, timezone
from typing import Any, Dict, List
import json
import httpx
from rich.console import Console
from rich.table import Table
from rich.json import JSON

# Configuration
GATEWAY_URL = os.getenv("GATEWAY_URL", "http://localhost:8085")
AUTH_TOKEN = os.getenv("AUTH_TOKEN", "changeme")

console = Console()


In [ ]:
def get_imessages(from_date: datetime, max_messages_per_thread: int | None = None) -> List[Dict[str, Any]]:
    """Fetch iMessages since the given date.
    
    Args:
        from_date: ISO 8601 datetime to fetch messages from
        max_messages_per_thread: Optional limit on number of messages per thread (None = no limit)
    """
    headers = {
        "Authorization": f"Bearer {AUTH_TOKEN}",
        "Content-Type": "application/json",
    }
    
    # Ensure datetime is timezone-aware (UTC)
    if from_date.tzinfo is None:
        from_date = from_date.replace(tzinfo=timezone.utc)
    else:
        from_date = from_date.astimezone(timezone.utc)
    
    params: Dict[str, Any] = {"from_date": from_date.isoformat()}
    if max_messages_per_thread is not None:
        params["max_messages_per_thread"] = max_messages_per_thread
    
    with httpx.Client(base_url=GATEWAY_URL, timeout=30.0) as client:
        response = client.get("/v1/imessages", params=params, headers=headers)
        response.raise_for_status()
        return response.json()


def print_results(threads: List[Dict[str, Any]]) -> None:
    """Pretty print the results."""
    console.print(f"\n[bold green]Found {len(threads)} thread(s)[/bold green]\n")
    
    for i, thread in enumerate(threads, 1):
        thread_id = thread.get("thread_id", "(no thread_id)")
        thread_name = thread.get("thread_name") or "(no name)"
        participants = thread.get("participants", [])
        messages = thread.get("messages", [])
        
        has_older_messages = thread.get("has_older_messages", False)
        
        console.print(f"[bold cyan]Thread {i}:[/bold cyan] {thread_name}")
        console.print(f"  Thread ID: {thread_id}")
        console.print(f"  Messages: {len(messages)}")
        if has_older_messages:
            console.print(f"  [dim italic]⚠ Older messages were truncated[/dim italic]")
        if participants:
            participants_str = ", ".join(participants)
            console.print(f"  Participants: [yellow]{participants_str}[/yellow]")
        else:
            console.print(f"  Participants: [dim](none)[/dim]")
        
        if messages:
            table = Table(show_header=True, header_style="bold magenta")
            table.add_column("Date", style="dim")
            table.add_column("From")
            table.add_column("Body", style="cyan", max_width=60)
            
            for msg in messages:
                date_str = msg.get("date", "")
                from_str = msg.get("from", "Unknown")
                body_str = msg.get("body", "")[:100]  # Truncate long messages
                if len(msg.get("body", "")) > 100:
                    body_str += "..."
                
                table.add_row(date_str, from_str, body_str)
            
            console.print(table)
        
        console.print()


## Test 1: Messages from last 30 days


In [3]:
# Get messages from the last 30 days
from_date = datetime.now(timezone.utc) - timedelta(days=2)
console.print(f"[bold]Fetching messages since:[/bold] {from_date.isoformat()}")

try:
    threads = get_imessages(from_date)
    print_results(threads)
except httpx.HTTPStatusError as e:
    console.print(f"[bold red]HTTP Error {e.response.status_code}:[/bold red] {e.response.text}")
except Exception as e:
    console.print(f"[bold red]Error:[/bold red] {e}")


Fetching messages since: 2025-12-16T19:54:36.842764+00:00

Found 22 thread(s)

Thread 1: (no name)

Thread ID: 2f41d658-a719-517b-a8b1-e6ae93c368ab

Messages: 1

...

Thread 22: (no name)

Thread ID: 69bc7666-23b2-534f-963a-8a3fd74f93a3

Messages: 1

Participants: +18333543471

┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Date                 ┃ From         ┃ Body                                                         ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 2025-12-18T14:09:32Z │ +18333543471 │ Lawadessa Cafe:  Holiday dessert pre-orders are officially   │
│                      │              │ open! Reserve your favorites now and enjo...                 │
└──────────────────────┴──────────────┴──────────────────────────────────────────────────────────────┘

## Test 2: Messages from last 7 days


In [ ]:
# Get messages from the last 7 days with max 10 messages per thread
from_date = datetime.now(timezone.utc) - timedelta(days=7)
console.print(f"[bold]Fetching messages since:[/bold] {from_date.isoformat()}")
console.print(f"[bold]Max messages per thread:[/bold] 10")

try:
    threads = get_imessages(from_date, max_messages_per_thread=10)
    print_results(threads)
except httpx.HTTPStatusError as e:
    console.print(f"[bold red]HTTP Error {e.response.status_code}:[/bold red] {e.response.text}")
except Exception as e:
    console.print(f"[bold red]Error:[/bold red] {e}")


## Test 3: Messages from a specific date


In [ ]:
# Get messages from a specific date (e.g., January 1, 2024)
from_date = datetime(2024, 1, 1, 0, 0, 0, tzinfo=timezone.utc)
console.print(f"[bold]Fetching messages since:[/bold] {from_date.isoformat()}")

try:
    threads = get_imessages(from_date)
    print_results(threads)
except httpx.HTTPStatusError as e:
    console.print(f"[bold red]HTTP Error {e.response.status_code}:[/bold red] {e.response.text}")
except Exception as e:
    console.print(f"[bold red]Error:[/bold red] {e}")


## Test 4: Raw JSON output


In [4]:
# Get raw JSON for inspection
from_date = datetime.now(timezone.utc) - timedelta(days=1)

try:
    threads = get_imessages(from_date)
    
    # Debug: Check if participants are in the response
    console.print(f"[bold]Total threads:[/bold] {len(threads)}")
    threads_with_participants = sum(1 for t in threads if t.get("participants"))
    console.print(f"[bold]Threads with participants:[/bold] {threads_with_participants}")
    
    # Show first thread's structure
    if threads:
        first_thread = threads[0]
        console.print(f"\n[bold]First thread structure:[/bold]")
        console.print(f"  thread_id: {first_thread.get('thread_id')}")
        console.print(f"  thread_name: {first_thread.get('thread_name')}")
        console.print(f"  participants: {first_thread.get('participants')}")
        console.print(f"  participants type: {type(first_thread.get('participants'))}")
        console.print(f"  participants count: {len(first_thread.get('participants', []))}")
    
    console.print("\n[bold]Full JSON:[/bold]")
    console.print(JSON(json.dumps(threads, indent=2, default=str)))
except httpx.HTTPStatusError as e:
    console.print(f"[bold red]HTTP Error {e.response.status_code}:[/bold red] {e.response.text}")
except Exception as e:
    console.print(f"[bold red]Error:[/bold red] {e}")


Total threads: 16

Threads with participants: 16

First thread structure:

thread_id: 8ea48b4b-55d1-519d-8ddb-fa3589a4bd3e

thread_name:

participants: ['Andrew Moring', 'Katie Patten']

participants type: <class 'list'>

participants count: 2

Full JSON:

[
  {
    "thread_id": "8ea48b4b-55d1-519d-8ddb-fa3589a4bd3e",
    "thread_name": "",
    "participants": [
      "Andrew Moring",
      "Katie Patten"
    ],
    "messages": [
      {
        "from": "Andrew Moring",
        "body": "[Image: Scene: A young girl in a turquoise shirt and jeans sits cross-legged on a white-painted 
bed with a red patterned cover in a bright, cozy room. Behind her is a wooden-framed window and radiator, with a… |
IMG_9014.heic][Image: A girl sits sideways on a blue balance stool in a living room, feet resting on the base, 
wearing a turquoise long-sleeve shirt and jeans. A younger child plays on the floor nearby; background shows a … | 
IMG_9016.heic]Playing Elf on the Shelf",
        "date": "2025-12-17T20:15:34Z"
      }
    ]
  },
  {
    "thread_id": "25c01035-7758-5830-ba75-de73e3822b76",
    "thread_name": "",
    "participants": [
      "Naomi Patten"
    ],
    "messages": [
      {
        "from": "Naomi Patten",
        "body": "I’m having trouble with my Ancestry account.",
        "date": "2025-12-17T20:17:15Z"
      },
      {
        "from": "Naomi Patten",
        "body": "I put all that in there so I didn’t have to carry it in my head…",
        "date": "2025-12-17T20:17:59Z"
      },
      {
        "from": "Chris Patten",
        "body": "Hahaha no worries just curiosity",
        "date": "2025-12-17T20:31:58Z"
      },
      {
        "from": "Naomi Patten",
        "body": "Did I share the account with you by any chance?",
        "date": "2025-12-17T20:34:51Z"
      },
      {
        "from": "Chris Patten",
        "body": "Nope",
        "date": "2025-12-17T21:08:08Z"
      }
    ]
  },
  {
    "thread_id": "4f69552d-0e3e-5337-bd80-384f54ed4522",
    "thread_name": "",
    "participants": [
      "John Patten"
    ],
    "messages": [
      {
        "from": "John Patten",
        "body": "Mom is having a meltdown over your request for information. She never did listen to me about 
ancestry.com and now seems to have at least 3 different accounts and cant find the original 1 she set up first. I 
am trying to talk her off the ledge now!",
        "date": "2025-12-17T20:25:00Z"
      },
      {
        "from": "Chris Patten",
        "body": "Hahhahahaha it was literally just curiosity",
        "date": "2025-12-17T20:31:44Z"
      },
      {
        "from": "John Patten",
        "body": "Now she’s deep in email passwords that she can’t find",
        "date": "2025-12-17T20:48:24Z"
      },
      {
        "from": "Chris Patten",
        "body": "Reacted 😂 to “Now she’s deep in email passwords that she can’t find”",
        "date": "2025-12-17T21:08:14Z"
      }
    ]
  },
  {
    "thread_id": "24ba5cdf-9dd8-520a-aec2-61ac7ba4c904",
    "thread_name": "",
    "participants": [
      "+19258993030"
    ],
    "messages": [
      {
        "from": "+19258993030",
        "body": "Leaf Blower (valued at $160). 🍂 Ideal for fall leaves!\nSupplies are limited—don't miss your 
chance!\n\negosleafblow.com/9v6cy7j��",
        "date": "2025-12-17T20:25:43Z"
      }
    ]
  },
  {
    "thread_id": "a20b12ce-60e0-5a22-b3e3-617b4145681d",
    "thread_name": "Whiskey first code later",
    "participants": [
      "Connor Tyrrell",
      "David Lozzi",
      "Josh Drumm",
      "Neil Mager"
    ],
    "messages": [
      {
        "from": "Neil Mager",
        "body": "So true, and now we know where irish coffee came from",
        "date": "2025-12-17T20:43:06Z"
      },
      {
        "from": "Josh Drumm",
        "body": "let's take a moment and thank the Irish for their contributions to society",
        "date": "2025-12-17T20:58:19Z"
      },
      {
        "from": "David Lozzi",
        "body": "Merry Christmas to me!",
        "date": "2025-12-17T20:59:21Z"
      },
      {
        "from": "David Lozzi",
        "body": "[Image: A hand holds a large bottle of 12-year-old whiskey. The amber spirit is visible through 
the glass, with burgundy and black labels featuring a prominent \"W

## Test 6: Limited messages per thread


In [ ]:
# Get messages with a limit on messages per thread
from_date = datetime.now(timezone.utc) - timedelta(days=30)
max_per_thread = 5

console.print(f"[bold]Fetching messages since:[/bold] {from_date.isoformat()}")
console.print(f"[bold]Max messages per thread:[/bold] {max_per_thread}")

try:
    threads = get_imessages(from_date, max_messages_per_thread=max_per_thread)
    
    # Show statistics
    total_messages = sum(len(t.get("messages", [])) for t in threads)
    console.print(f"\n[bold]Results:[/bold]")
    console.print(f"  Total threads: {len(threads)}")
    console.print(f"  Total messages: {total_messages}")
    console.print(f"  Average messages per thread: {total_messages / len(threads) if threads else 0:.1f}")
    
    # Show which threads have truncated messages
    threads_truncated = [t for t in threads if t.get("has_older_messages", False)]
    if threads_truncated:
        console.print(f"\n[bold]Threads with truncated messages:[/bold] {len(threads_truncated)}")
        for thread in threads_truncated[:5]:  # Show first 5
            console.print(f"  - {thread.get('thread_name') or '(no name)'}: {len(thread.get('messages', []))} messages shown (older messages truncated)")
    
    print_results(threads)
except httpx.HTTPStatusError as e:
    console.print(f"[bold red]HTTP Error {e.response.status_code}:[/bold red] {e.response.text}")
except Exception as e:
    console.print(f"[bold red]Error:[/bold red] {e}")


## Test 5: Statistics


In [ ]:
# Get statistics about the messages
from_date = datetime.now(timezone.utc) - timedelta(days=30)

try:
    threads = get_imessages(from_date)
    
    total_threads = len(threads)
    total_messages = sum(len(t.get("messages", [])) for t in threads)
    
    # Count unique senders
    all_senders = set()
    for thread in threads:
        for msg in thread.get("messages", []):
            all_senders.add(msg.get("from", "Unknown"))
    
    # Count unique participants across all threads
    all_participants = set()
    threads_with_participants = 0
    for thread in threads:
        participants = thread.get("participants", [])
        if participants:
            threads_with_participants += 1
            for participant in participants:
                all_participants.add(participant)
    
    # Find earliest and latest messages
    all_dates = []
    for thread in threads:
        for msg in thread.get("messages", []):
            date_str = msg.get("date")
            if date_str:
                try:
                    all_dates.append(datetime.fromisoformat(date_str.replace("Z", "+00:00")))
                except:
                    pass
    
    console.print("\n[bold]Statistics:[/bold]")
    console.print(f"  Total threads: {total_threads}")
    console.print(f"  Total messages: {total_messages}")
    console.print(f"  Unique senders: {len(all_senders)}")
    console.print(f"  Threads with participants: {threads_with_participants}")
    console.print(f"  Unique participants: {len(all_participants)}")
    if all_dates:
        console.print(f"  Earliest message: {min(all_dates).isoformat()}")
        console.print(f"  Latest message: {max(all_dates).isoformat()}")
    
    if all_senders:
        console.print("\n[bold]Senders:[/bold]")
        for sender in sorted(all_senders):
            console.print(f"  - {sender}")
    
    if all_participants:
        console.print("\n[bold]Participants:[/bold]")
        for participant in sorted(all_participants):
            console.print(f"  - {participant}")
    
except httpx.HTTPStatusError as e:
    console.print(f"[bold red]HTTP Error {e.response.status_code}:[/bold red] {e.response.text}")
except Exception as e:
    console.print(f"[bold red]Error:[/bold red] {e}")
